# Model Design and Training

Train baseline Random Forest classifiers and explore River-based incremental classifiers.

In [ ]:
# Cell 1: Setup
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve
import matplotlib.pyplot as plt
import seaborn as sns

# Load feature data
df = pd.read_csv('data/processed/btc_features.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

print("Dataset Shape:", df.shape)
print(f"Crash Rate: {df['crash'].mean()*100:.2f}%")

In [ ]:
# Cell 2: Prepare data
# Get feature columns
feature_cols = [col for col in df.columns if col not in 
                ['timestamp', 'crash', 'open', 'high', 'low', 'close', 'volume',
                 'open_time', 'close_time', 'quote_asset_volume', 'num_trades',
                 'taker_buy_base', 'taker_buy_quote', 'ignore']]

X = df[feature_cols].fillna(0)
y = df['crash']

print(f"Features: {len(feature_cols)}")
print(f"Samples: {len(X):,}")
print(f"Positive class (crashes): {y.sum():,} ({y.mean()*100:.2f}%)")
print(f"Negative class (normal): {(~y.astype(bool)).sum():,} ({(1-y.mean())*100:.2f}%)")

# Time-based split (last 20% for test)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"\nTrain set: {len(X_train):,} samples")
print(f"Test set: {len(X_test):,} samples")
print(f"Train crash rate: {y_train.mean()*100:.2f}%")
print(f"Test crash rate: {y_test.mean()*100:.2f}%")

In [ ]:
# Cell 3: Train baseline RF model
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import time

print("Training baseline Random Forest model...")
start_time = time.time()

# Train model
rf_baseline = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=100,
    class_weight='balanced',  # Handle imbalance
    random_state=42,
    n_jobs=-1
)

rf_baseline.fit(X_train, y_train)

training_time = time.time() - start_time
print(f"Training completed in {training_time:.2f} seconds")

# Predictions
y_pred = rf_baseline.predict(X_test)
y_pred_proba = rf_baseline.predict_proba(X_test)[:, 1]

# Evaluation
print("\n" + "=" * 60)
print("BASELINE MODEL PERFORMANCE")
print("=" * 60)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Crash']))

print(f"\nROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix - Baseline RF', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('docs/figures/baseline_confusion_matrix.png', dpi=300)
plt.show()